# Tracking error analysis

Consumes a rosbag recorded by `launch/figure8.launch.py record:=true` and produces the
tracking-error figures for the README and for `docs/TUNING_GUIDE.md`.

Outputs:
- `media/tracking_error_figure8.png` — position error vs time, per axis, with RMS annotated
- `media/tracking_xy.png` — commanded vs flown ground track
- `media/tracking_error_vs_accel.png` — the §12.2 diagnostic plot
- a markdown summary table pasted into the README

**Honesty rules for this notebook** (a reviewer will read the plots before the code):
- always plot the *reference the controller was actually given* (`/nmpc_node/reference_path`,
  the first stage of each published horizon), not the ideal analytic curve
- state the wind/disturbance condition in every title
- never crop the transient out of a plot without saying so in the caption


## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Make analysis/ importable regardless of the kernel's cwd, then pull in the shared bag
# helpers — type registration and bag iteration live in ONE place (bag_utils.py).
if Path.cwd().name == "analysis":
    sys.path.insert(0, str(Path.cwd()))
else:
    sys.path.insert(0, str(Path.cwd() / "analysis"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

from bag_utils import read_topics, require_topic

# --- re-runnable by editing this one cell ---------------------------------------------------
BAG_PATH = Path("output/figure8_20260101_120000")   # change me: analysis/output/figure8_<ts>
OUT_DIR = Path("../media")                           # figures land next to the README
OUT_DIR.mkdir(parents=True, exist_ok=True)

TAKEOFF_ALTITUDE = 1.5        # [m] AGL, matches trajectory_params.yaml `altitude`
DISTURBANCE = "no wind"       # state the disturbance in every title (§12.2)


## 2. Load the bag

In [ ]:
data = read_topics(BAG_PATH, (
    "/nmpc_node/status",
    "/nmpc_node/reference_path",            # the reference the controller was actually given
    "/fmu/out/vehicle_local_position",      # flown position, PX4 local frame (NED)
    "/fmu/in/vehicle_attitude_setpoint",    # what the controller sent PX4
))
for topic in data:
    require_topic(data, topic, BAG_PATH)

# Time base: the BAG RECORD timestamp (ns) — the same recorder clock for every topic. The
# PX4 `timestamp` (µs since power-on) and the node's `header.stamp` (ROS time) are NOT
# mutually aligned, so the record time is the single base we trust; merge_asof is used
# wherever two topics must be combined (§12.2). In SITL the record time == ROS sim time.

status = pd.DataFrame([
    {"t": t, "state": m.controller_state,
     "ex": m.position_error.x, "ey": m.position_error.y, "ez": m.position_error.z,
     "pn": m.position_error_norm, "yaw_err": m.yaw_error,
     "solve_ms": m.solver.wall_time_ms, "solver_status": m.solver.status,
     "T": m.collective_thrust_newton,
     "qw": m.attitude_setpoint.w, "qx": m.attitude_setpoint.x,
     "qy": m.attitude_setpoint.y, "qz": m.attitude_setpoint.z}
    for t, m in data["/nmpc_node/status"]
])

local = pd.DataFrame([
    {"t": t, "x": m.x, "y": m.y, "z": m.z, "xy_valid": m.xy_valid, "z_valid": m.z_valid}
    for t, m in data["/fmu/out/vehicle_local_position"]
])

# reference_path carries a full horizon per message; the FIRST pose is the reference the
# controller was tracking at that instant.
ref = pd.DataFrame([
    {"t": t, "rx": m.poses[0].pose.position.x,
     "ry": m.poses[0].pose.position.y, "rz": m.poses[0].pose.position.z}
    for t, m in data["/nmpc_node/reference_path"] if m.poses
])

att_sp = pd.DataFrame([
    {"t": t, "qw": m.q_d[0], "qx": m.q_d[1], "qy": m.q_d[2], "qz": m.q_d[3]}
    for t, m in data["/fmu/in/vehicle_attitude_setpoint"]
])

print(f"status {len(status)} msgs · local {len(local)} · reference {len(ref)} · "
      f"att_sp {len(att_sp)}")


def asof(left: pd.DataFrame, right: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """merge_asof on the record clock, 50 ms tolerance (two control ticks at 100 Hz)."""
    return pd.merge_asof(left.sort_values("t"), right.sort_values("t"), on="t",
                         direction="nearest", tolerance=50_000_000, suffixes=("", "_r"))


## 3. Error time series

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
t_s = (status["t"] - status["t"].iloc[0]) / 1e9

for ax, (col, name) in zip(axes, [("ex", "x"), ("ey", "y"), ("ez", "z")]):
    err = status[col]
    ax.plot(t_s, err, lw=0.5, color="tab:blue")
    # rolling mean +/- 1 sigma band, 0.5 s window at ~100 Hz
    win = 50
    mean = err.rolling(win, center=True).mean()
    sd = err.rolling(win, center=True).std()
    ax.fill_between(t_s, mean - sd, mean + sd, alpha=0.25, color="tab:orange",
                    label="±1σ band")
    rms = np.sqrt(np.mean(err ** 2))
    pk = np.max(np.abs(err))
    ax.axhline(0.0, color="k", lw=0.4)
    ax.set_ylabel(f"{name} error [m]")
    ax.annotate(f"RMS {rms:.3f} m   max {pk:.3f} m", xy=(0.99, 0.95), ha="right",
                va="top", xycoords="axes fraction", fontsize=9)
    ax.legend(loc="lower right", fontsize=8)

# phase boundaries from the controller state machine (takeoff/tracking/landing)
for state, name in ((2, "takeoff"), (3, "tracking"), (4, "landing")):
    t_state = status.loc[status["state"] == state, "t"]
    if not t_state.empty:
        for ax in axes:
            ax.axvline((t_state.iloc[0] - status["t"].iloc[0]) / 1e9,
                       ls=":", color="gray", alpha=0.7)
axes[-1].set_xlabel("time [s] (bag record clock, takeoff not cropped)")
fig.suptitle(f"tracking error — figure-8, {DISTURBANCE}; full flight, no transient cropped")
fig.tight_layout()
fig.savefig(OUT_DIR / "tracking_error_figure8.png", dpi=150, bbox_inches="tight")
print("wrote", OUT_DIR / "tracking_error_figure8.png")


## 4. Ground track

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

# flown track — PX4 local frame is NED; the XY plane is the same as ENU up to handedness,
# and the 8 is symmetric in x and y, so the shape is directly comparable.
ax.plot(local["x"], local["y"], lw=0.8, color="tab:blue", label="flown (local_position)")

# reference actually given: first pose of each published Path message (§12.2)
ax.plot(ref["rx"], ref["ry"], "--", lw=1.2, color="tab:orange",
        label="reference (first stage of each horizon)")

# colour-grade the flown track by speed
dt_s = local["t"].diff() / 1e9
speed = np.hypot(local["x"].diff(), local["y"].diff()) / dt_s
sc = ax.scatter(local["x"], local["y"], c=speed, s=4, cmap="viridis", alpha=0.6)
plt.colorbar(sc, ax=ax, label="speed [m/s]")

ax.set_aspect("equal")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.legend(loc="upper right")
ax.set_title(f"commanded vs flown ground track — figure-8, {DISTURBANCE}")
fig.tight_layout()
fig.savefig(OUT_DIR / "tracking_xy.png", dpi=150, bbox_inches="tight")
print("wrote", OUT_DIR / "tracking_xy.png")


## 5. Error vs commanded acceleration

In [ ]:
# The most informative figure in the notebook (§12.2): |position error| against
# |commanded acceleration|. The commanded acceleration is computed from the telemetry the
# controller actually sent — attitude setpoint quaternion + collective thrust:
#     a_cmd = (T/m) * R(q_d) e_z - g e_z     (world ENU)
# Mass comes from the same params/x500_calibration.yaml the node reads.
import yaml

airframe = yaml.safe_load(
    Path("../uav_mpc/params/x500_calibration.yaml").read_text())["airframe"]
m, g = float(airframe["mass"]), 9.80665

qw, qx, qy, qz = (status[c].to_numpy() for c in ("qw", "qx", "qy", "qz"))
# R e_z for an active wxyz quaternion: [2(qx qz + qw qy), 2(qy qz - qw qx), 1 - 2(qx²+qy²)]
Rz = np.stack([2 * (qx * qz + qw * qy),
               2 * (qy * qz - qw * qx),
               1 - 2 * (qx ** 2 + qy ** 2)], axis=1)
a = (status["T"].to_numpy()[:, None] / m) * Rz - np.array([0.0, 0.0, g])
a_h = np.hypot(a[:, 0], a[:, 1])
e = status["pn"].to_numpy()
mask = np.isfinite(a_h) & np.isfinite(e) & (e < 5.0)  # clip the takeoff/landing spikes

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(a_h[mask], e[mask], s=4, alpha=0.35, color="tab:blue")
if mask.sum() > 10:
    slope, intercept = np.polyfit(a_h[mask], e[mask], 1)
    xs = np.linspace(a_h[mask].min(), a_h[mask].max(), 50)
    ax.plot(xs, slope * xs + intercept, "r-", lw=1.5,
            label=f"linear fit, slope {slope:.4f} s²")
    ax.set_title(f"error vs commanded acceleration — figure-8, {DISTURBANCE}\n"
                 f"slope {slope:.3f} ⇒ error grows with acceleration (model deficiency); "
                 f"flat ⇒ tuning/latency (§12.2)")
ax.set_xlabel("|commanded horizontal acceleration| [m/s²]")
ax.set_ylabel("|position error| [m]")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "tracking_error_vs_accel.png", dpi=150, bbox_inches="tight")
print("wrote", OUT_DIR / "tracking_error_vs_accel.png")


## 6. Summary table

In [ ]:
# Markdown summary table for the README / TUNING_GUIDE (df.to_markdown, with a fallback
# for environments without `tabulate`).
dt_s = local["t"].diff() / 1e9
speed = np.hypot(local["x"].diff(), local["y"].diff()) / dt_s

# laps: the 8 crosses the centre twice per lap; count y sign changes and halve
signs = np.sign(local["y"].to_numpy())
crossings = int(np.sum(signs[1:] * signs[:-1] < 0))
laps = round(crossings / 2)

qw, qx, qy, qz = (status[c].to_numpy() for c in ("qw", "qx", "qy", "qz"))
tilt = np.degrees(np.arccos(np.clip(1 - 2 * (qx ** 2 + qy ** 2), -1, 1)))

summary = pd.DataFrame([{
    "condition": DISTURBANCE,
    "rms_error_x_m": float(np.sqrt(np.mean(status["ex"] ** 2))),
    "rms_error_y_m": float(np.sqrt(np.mean(status["ey"] ** 2))),
    "rms_error_z_m": float(np.sqrt(np.mean(status["ez"] ** 2))),
    "max_error_m": float(status["pn"].max()),
    "yaw_rms_deg": float(np.degrees(np.sqrt(np.mean(status["yaw_err"] ** 2)))),
    "laps": laps,
    "mean_speed_m_s": float(np.nanmean(speed)),
    "peak_speed_m_s": float(np.nanmax(speed)),
    "peak_tilt_deg": float(np.nanmax(tilt)),
    "solver_failures": int((status["solver_status"] != 0).sum()),
}])


def to_md(df: pd.DataFrame) -> str:
    try:
        return df.T.to_markdown()
    except ImportError:  # tabulate not installed
        lines = [df.T.reset_index().to_csv(sep="|", index=False)]
        return "\n".join(lines)


table = to_md(summary)
print(table)
